In [52]:
from pandas import read_csv

def read_data(file_path, num_features = 2, have_time = False):
    series_influ_A_df = read_csv(file_path, engine='python')
    series_influ_A_df = series_influ_A_df.rename(columns= {"Influenza A - All types of surveillance": "case"})

    # because since 2011-03-01 It was announced that the H1N12009 flu had been controlled and treated as regular seasonal flu and
    # since 2020-02-01, it's time for covid 
    
    series_influ_A_df = series_influ_A_df.loc [(series_influ_A_df['Month'] >='2011-04-01') & (series_influ_A_df['Month'] <='2020-02-01')]
    if not have_time:
        return series_influ_A_df.dropna()[["case", "temp", "dew", "tempmax", "humidity","tempmin","windspeed"][:num_features]]
    return series_influ_A_df.dropna()[["Month", "case", "temp", "dew", "tempmax", "humidity","tempmin","windspeed"][:num_features+1]]

file_path = '../../temp_data/influA_vietnam_temp_month.csv'
#Load data set
df = read_data(file_path,7)

In [53]:
import numpy as np
# def multi_corr(rxz , ryz, rxy):
#     a= np.sqrt((rxz*rxz + ryz*ryz -2*rxz*ryz*rxy)/ (1-rxy*rxy))
#     print(a)
#     return a

def multi_corr(df, dependent_var, independent_vars):
    corr_matrix = df.corr()
    
    # Extract submatrices
    R = corr_matrix.loc[[dependent_var] + independent_vars , [dependent_var] + independent_vars ].values
    rxz = R[0,1]
    ryz = R[0,2]
    rxy = R[1,2]
    return np.sqrt((rxz*rxz + ryz*ryz -2*rxz*ryz*rxy)/ (1-rxy*rxy))



In [50]:
# a = corr_matrix.loc[['case'] + ['temp', 'dew'], ['case'] + ['temp', 'dew']]
# a

In [57]:
# x = 1
# y = 2
# z = 0
list_independents = [
    ['temp', 'dew'], ['temp', 'tempmax'],
    ['temp', 'humidity'], ['temp', 'windspeed'],
    ['temp', 'tempmin'],['humidity', 'dew'],
    ['windspeed', 'dew'],['tempmin', 'dew'],
    ['tempmax', 'dew'], ['tempmax', 'tempmin'],
    ['tempmax', 'humidity'], ['tempmax', 'windspeed'],
    ['humidity', 'tempmin'],['humidity', 'windspeed'],
    ['windspeed', 'tempmin'],
    ]

for independent_vars in list_independents:
    rcase_temp_dew = multi_corr(df, 'case', independent_vars)
    print(independent_vars, "=>", rcase_temp_dew)
# rcase_temp_dew

['temp', 'dew'] => 0.48141447906880985
['temp', 'tempmax'] => 0.4814465882553716
['temp', 'humidity'] => 0.48211531715792927
['temp', 'windspeed'] => 0.4788721150926929
['temp', 'tempmin'] => 0.47858638573867784
['humidity', 'dew'] => 0.48361140129261465
['windspeed', 'dew'] => 0.45091947393020415
['tempmin', 'dew'] => 0.46509918153891294
['tempmax', 'dew'] => 0.47164547685819724
['tempmax', 'tempmin'] => 0.4777787090213494
['tempmax', 'humidity'] => 0.470355298554902
['tempmax', 'windspeed'] => 0.4682885239646308
['humidity', 'tempmin'] => 0.4758955984715409
['humidity', 'windspeed'] => 0.1743492815156416
['windspeed', 'tempmin'] => 0.46729514425725954


In [43]:
import numpy as np
import pandas as pd

def multiple_correlation_coefficient(df, dependent_var, independent_vars):
    """
    Compute the multiple correlation coefficient of a dependent variable with a set of independent variables.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing the variables
    dependent_var (str): Name of the dependent variable
    independent_vars (list): List of names of the independent variables
    
    Returns:
    float: Multiple correlation coefficient
    """
    # Compute the correlation matrix
    corr_matrix = df.corr()
    
    # Extract submatrices
    R = corr_matrix.loc[independent_vars + [dependent_var], independent_vars + [dependent_var]].values
    R_zz = R[-1, -1]
    R_zx = R[-1, :-1]
    R_xx = R[:-1, :-1]
    
    # Compute the multiple correlation coefficient R^2
    R2 = R_zz - np.dot(np.dot(R_zx, np.linalg.inv(R_xx)), R_zx.T)
    R2 = 1 - R2
    
    # Compute the multiple correlation coefficient R
    R = np.sqrt(R2)
    
    return R

# Example usage:
data = {
    'x': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'y': [2, 4, 6, 8, 10, 12, 14, 16, 18, 20],
    'z': [10, 9, 8, 7, 6, 5, 4, 3, 2, 1],
    'w': [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
}

df = pd.DataFrame(data)

dependent_var = 'x'
independent_vars = ['y', 'z', 'w']

R = multiple_correlation_coefficient(df, dependent_var, independent_vars)
print("Multiple Correlation Coefficient R:", R)


LinAlgError: Singular matrix